In [1]:
!pip install camel_tools

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 12.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 97.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 15.0 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=e9b2252f67ad009d83ee7e52d52e9fdb847742053483a923099f868db8d9fce8
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: sciki

In [2]:
import pandas as pd
import numpy as np
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import normalize_alef_ar
from IPython.display import display,Markdown
from tqdm.auto import tqdm
import ast
import re

tqdm.pandas()

# Make the db


In [11]:
df = pd.read_parquet('data/sunnah.parquet')
df.head()

,book_id,book_category,book_type,book_date,book_name,category_name,all_authors,main_author_death,content,page_count
611,642,9,1,279,كتاب العلل الواقع بآخر جامع الترمذي - ت بشار,العلل والسؤلات الحديثية,الترمذي,279,"[{""page_id"": ""642-1"", ""body"": ""<span data-type...",82
767,805,6,1,311,الحث على التجارة - من «الجامع» للخلال - ت العوضي,كتب السنة,أبو بكر الخلال,311,"[{""page_id"": ""805-1"", ""body"": """", ""header"": """"...",77
984,1077,6,1,311,السنة لأبي بكر بن الخلال,كتب السنة,أبو بكر الخلال,311,"[{""page_id"": ""1077-1"", ""body"": ""<span data-typ...",2004
1209,1362,6,1,256,بر الوالدين - البخاري - ت مكي,كتب السنة,البخاري,256,"[{""page_id"": ""1362-1"", ""body"": """", ""header"": ""...",351
1471,1733,6,1,360,المعجم الكبير للطبراني,كتب السنة,الطبراني,360,"[{""page_id"": ""1733-1"", ""body"": ""<span data-typ...",31863


In [12]:
df['content'] = df['content'].apply(ast.literal_eval)
df = df.explode('content').reset_index(drop=True)

In [13]:
df['headers'] = df['content'].progress_apply(lambda x: x.get('headers',''))
df['page'] = df['content'].progress_apply(lambda x: int(x['page_id'].split('-')[-1]))
df['body'] = df['content'].progress_apply(lambda x: str(x['body']))
df = df[df['body'] != '']
# df = df.progress_apply(adding_page_tobody,axis=1)
df.head()

100%|██████████| 144831/144831 [00:01<00:00, 133773.42it/s]


,book_id,book_category,book_type,book_date,book_name,category_name,all_authors,main_author_death,content,page_count,headers,page,body
0,642,9,1,279,كتاب العلل الواقع بآخر جامع الترمذي - ت بشار,العلل والسؤلات الحديثية,الترمذي,279,"{'page_id': '642-1', 'body': '<span data-type=...",82,[كتاب العلل],1,"<span data-type=""title"" id=toc-1>كِتَابُ الْعِ..."
1,642,9,1,279,كتاب العلل الواقع بآخر جامع الترمذي - ت بشار,العلل والسؤلات الحديثية,الترمذي,279,"{'page_id': '642-2', 'body': 'وَمَا ذَكَرْنَا ...",82,[كتاب العلل],2,وَمَا ذَكَرْنَا فِي هَذَا الْكِتَابِ مِنِ اخْت...
2,642,9,1,279,كتاب العلل الواقع بآخر جامع الترمذي - ت بشار,العلل والسؤلات الحديثية,الترمذي,279,"{'page_id': '642-3', 'body': 'وَمَا كَانَ مِنَ...",82,[كتاب العلل],3,وَمَا كَانَ مِنَ الْوُضُوءِ وَالصَّلَاةِ فَحَد...
3,642,9,1,279,كتاب العلل الواقع بآخر جامع الترمذي - ت بشار,العلل والسؤلات الحديثية,الترمذي,279,"{'page_id': '642-4', 'body': 'وَإِنَّمَا حَمَل...",82,[كتاب العلل],4,وَإِنَّمَا حَمَلَنَا عَلَى مَا بَيَّنَّا فِي ه...
4,642,9,1,279,كتاب العلل الواقع بآخر جامع الترمذي - ت بشار,العلل والسؤلات الحديثية,الترمذي,279,"{'page_id': '642-5', 'body': 'يُبَيِّنُوا ضَعْ...",82,[كتاب العلل],5,يُبَيِّنُوا ضَعْفَ هَؤُلَاءِ؛ لِكَيْ يُعْرَفُو...


In [14]:
# random = np.random.randint(1,len(df))
# x = df.iloc[random]
# print(f'Book id: {x['book_name']}')
# print(f'Book id: {x['headers']}')
# display(Markdown(x['body']))
# print(f'Book page:{x['book_id']}/{x['page']}')
# display(x['body'])
# display('---')
# text = x['body'].replace('\r','\n')

# # patt = r'(?ms)^.*?([٠-٩]+)\s*-\s*.*?(?=^.*?[٠-٩]+\s*-|\Z)'
# patt = r'(?s)[0-9٠-٩]+\s*-\s*.*?(?=(?:(?<=[\s.)])\s*[0-9٠-٩]+\s*-\s*)|\Z)'

# pattern = re.compile(
#     patt,re.M | re.S
# )

# for match in pattern.finditer(text):
#     display(Markdown(match.group()))


In [15]:
def adding_page_tobody(row):
    row['body'] = f"{row.body} <<{row.page}>>"   
    return row
df = df.progress_apply(adding_page_tobody,axis=1)

  0%|          | 0/144516 [00:00<?, ?it/s]

100%|██████████| 144516/144516 [00:30<00:00, 4802.41it/s] 


In [16]:
df = df.drop(columns=['content'])

In [17]:
from bs4 import BeautifulSoup as bs
from bs4 import NavigableString

def get_html(row):
    soup = bs(row.body, "html.parser")

    spans = soup.find_all(attrs={"data-type": "title"})
    if not spans:
        return row

    headers = row.headers

    for span in spans:
        body = "".join(
            str(c) if isinstance(c, NavigableString) else c.get_text()
            for c in span.contents
        ).strip()

        try:
            idx = headers.index(body)
            depth = idx + 1
        except ValueError:
            depth = len(headers)

        markdown = "#" * depth + " " + body + "\n"
        span.replace_with(NavigableString(markdown))

    # Remove every remaining HTML tag, keeping only their text
    for tag in soup.find_all(True):
        tag.unwrap()

    row["body"] = soup.get_text()
    return row

df = df.progress_apply(get_html,axis=1)

100%|██████████| 144516/144516 [00:52<00:00, 2756.45it/s]


In [19]:
def merge_text(texts):
    result = []

    for i, text in enumerate(texts):
        result.append(text)

        if i == len(texts) - 1:
            continue

        next_text = texts[i + 1]

        if re.match(r"^#{1,6}\s", next_text):
            result.append("\n\n")
        else:
            result.append(" ")

    return "".join(result)

grouped = (
    df.groupby(df.columns[:-3].tolist(), as_index=False)
      .agg(body=("body", lambda s: merge_text(s.tolist())))
)

In [20]:
import re

HEADER_RE = re.compile(r'^(#{1,6})\s+(.*)$')

def iter_sections(text):
    current = [None] * 6
    body = []

    for line in text.splitlines():
        m = HEADER_RE.match(line)

        if m:
            if body:
                yield {
                    "headers": [h for h in current if h],
                    "body": "\n".join(body).strip()
                }
                body = []

            level = len(m.group(1))
            current[level - 1] = m.group(2).strip()

            for i in range(level, 6):
                current[i] = None
        else:
            body.append(line)

    if body:
        yield {
            "headers": [h for h in current if h],
            "body": "\n".join(body).strip()
        }

In [21]:
rows = []

for _, row in tqdm(grouped.iterrows()):
    for section in iter_sections(row["body"]):
        rows.append({
            "book_id": row["book_id"],
            "'book_category'": row["book_category"],
            "book_type": row["book_type"],
            "book_date": row["book_date"],
            "book_name": row["book_name"],
            "category_name": row["category_name"],
            "all_authors": row["all_authors"],
            "book_name": row["book_name"],
            "main_author_death": row["main_author_death"],
            "page_count": row["page_count"],
            "headers": " / ".join(section["headers"]),
            "body": section["body"],
        })

sections_df = pd.DataFrame(rows)

116it [00:02, 52.85it/s]


by page: 642, 8271

by []: 5945, 8601

remove: 6655 36080 9697 117359

## fixex

In [22]:
sections_df = sections_df[~sections_df['book_id'].isin([6655,36080,9697,642, 8271,117359])]

### by []

In [23]:
def fix_brak(text):
    start_pattern = re.compile(
    r'(?:\[[٠-٩]+\]|[٠-٩]+)'
)
    return re.sub(start_pattern, r'\g<0> - ', text)

mask = sections_df['book_id'].isin([5945, 8601])

sections_df.loc[mask, 'body'] = sections_df.loc[mask, 'body'].apply(fix_brak)

In [24]:
sections_df[sections_df['book_id'].isin([5945, 8601])]['body'].iloc[0]

'[١] -  قَالَ سَمِعت أَحْمد يَقُول روى سعيد من بَاعَ عبدا وَله مَال عَن قَتَادَة عَن عِكْرِمَة عَن ابْن عمر وَرَوَاهُ هِشَام وَهَمَّام عَن عِكْرِمَة وَهُوَ ابْن خَالِد عَن الزُّهْرِيّ <<1>> [٢] -  قلت لِأَحْمَد سعيد بن أبي عرُوبَة عَن أَيُّوب عَن نَافِع عَن ابْن عمر قَالَ كُنَّا نمسح وَنحن مَعَ نَبينَا قَالَ أسأَل الله عَافِيَة <<2>> فَقلت شُعَيْب بن إِسْحَاق قَالَ شُعَيْب سمع مِنْهُ بآخر رَمق\nقَالَ الْحُسَيْن يَعْنِي أَن شُعَيْب بن إِسْحَاق سمع من سعيد بن أبي عرُوبَة هَذَا الحَدِيث بآخر رَمق\n[٣] -  قلت لِأَحْمَد يرْوى من حَدِيث [الزُّهْرِيّ] عَن عُثْمَان الْإِيلَاء <<3>> تَطْلِيقَة فَأنكرهُ فَقَالَ الزُّهْرِيّ إِنَّمَا يروي عَن أبي بكر وَسَعِيد [وَقبيصَة] هُوَ ابْن ذُؤَيْب وَالنَّاس يروون عَن عَطاء الْخُرَاسَانِي عَن <<4>> سعيد ورأي سعيد بن الْمسيب خلاف لذاك الَّذِي يروي\n[٤] -  قَالَ سَمِعت أَحْمد وَذكر حَدِيث ابْن عَبَّاس فِي صَلَاة الْكُسُوف ان عبد الرَّحْمَن قَالَ كَذَا كَذَا رَكْعَة فِيهِ وَكَانَ وَكِيع يُخَالِفهُ فَعرض عَلَيْهِ يَعْنِي على وَكِيع بعد ذَلِك فَرجع عَنهُ صَار إِل

### by page

In [63]:
pages_df = df[df['book_id'].isin([642, 8271])]

# Split

In [26]:
# page = 1
def splitter(text):
    global page
    text = text.replace('\r','\n')

    # patt = r'(?ms)^.*?([٠-٩]+)\s*-\s*.*?(?=^.*?[٠-٩]+\s*-|\Z)'
    patt = r"""(?sx)
        (?:\[[0-9٠-٩]+\]|[0-9٠-٩]+)\s*-\s*
        .*?
        (?=
            (?:
                (?<=[\s.)])
                \s*
                (?:\[[0-9٠-٩]+\]|[0-9٠-٩]+)\s*-\s*
            )
            |\Z
        )"""

    pattern = re.compile(
        patt,re.M | re.S
    )

    return pattern.findall(text)
        

In [55]:
sections_df_exp = sections_df.copy()

In [56]:
sections_df_exp['body'] = sections_df_exp['body'].progress_apply(splitter) 

100%|██████████| 24281/24281 [00:05<00:00, 4319.02it/s] 


In [57]:
sections_df_exp = sections_df_exp.explode('body')

In [58]:
books = []
for book in sections_df_exp[sections_df_exp.isna().any(axis=1)]['book_id'].unique():
    if len(sections_df_exp[sections_df_exp['book_id']==book])<30:
        books.append(book)
        
books

[np.int64(6404), np.int64(13077)]

In [59]:
sections_df_exp = sections_df_exp.dropna()
sections_df_exp = sections_df_exp.drop_duplicates()

In [64]:
page_re = re.compile(r'<<(\d+)>>')
athar_num_re = re.compile(r'(?:\[([٠-٩0-9]+)\]|([٠-٩0-9]+))')

pages = []
athar_nums = []
bodies = []

page = 1
book = sections_df_exp['book_id'].iat[0]

for row in tqdm(sections_df_exp.itertuples(),total=len(sections_df_exp)):
    if row.book_id != book:
        book = row.book_id
        page = 1

    matches = page_re.findall(row.body)

    if not matches:
        pages.append(page)
    elif len(matches) > 1:
        pages.append(", ".join(matches))
        page = matches[-1]
    else:
        pages.append(matches[0])
        page = matches[0]

    m = athar_num_re.search(row.body)
    athar_nums.append(m.group(0) if m else None)

    # bodies.append(page_re.sub("", row.body))

sections_df_exp["page"] = pages
sections_df_exp["athar_num"] = athar_nums
# sections_df_exp["body"] = bodies

100%|██████████| 110880/110880 [00:08<00:00, 13284.83it/s]


In [65]:
sections_df_exp.columns = ['book_id',
 "book_category",
 'book_type',
 'book_date',
 'book_name',
 'category_name',
 'all_authors',
 'main_author_death',
 'page_count',
 'headers',
 'body',
 'athar_num',
 'page']

In [66]:
pages_df['headers'] = pages_df['headers'].apply(lambda x: " / ".join(x))

In [67]:
pages_df['athar_num'] = pages_df['page']

In [70]:
sections_df_exp.columns

Index(['book_id', 'book_category', 'book_type', 'book_date', 'book_name',
       'category_name', 'all_authors', 'main_author_death', 'page_count',
       'headers', 'body', 'athar_num', 'page'],
      dtype='str')

In [83]:
full = pd.concat([sections_df_exp,pages_df])

In [85]:
arabic_digits = str.maketrans(
    '٠١٢٣٤٥٦٧٨٩',
    '0123456789'
)

full["athar_num"] = full["athar_num"].str.translate(arabic_digits)

In [91]:
full.head()

,book_id,book_category,book_type,book_date,book_name,category_name,all_authors,main_author_death,page_count,headers,body,athar_num,page
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"١ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" مَنْ قَالَ: ا...",1,1
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"٢ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" إِذَا قَالَ ا...",2,2
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"٣ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" مَنْ قَالَ: ا...",3,2
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"٤ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" مَنْ قَالَ ذَ...",4,2
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,٥ - سَأَلْتُ أَبِي ﵀ عَنِ الصَّلَاةِ، خَلْفَ أ...,5,2


In [87]:
full['page'] = full['page'].astype(str)
full["athar_num"] = (full["athar_num"].fillna(-1).astype(str).str.strip("[]").astype(int))
full['athar_num'] = full['athar_num'].astype(int)

In [90]:
full.info()

<class 'pandas.DataFrame'>
Index: 110984 entries, 0 to 57928
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   book_id            110984 non-null  uint32
 1   book_category      110984 non-null  uint8 
 2   book_type          110984 non-null  uint8 
 3   book_date          110984 non-null  uint16
 4   book_name          110984 non-null  str   
 5   category_name      110984 non-null  str   
 6   all_authors        110984 non-null  str   
 7   main_author_death  110984 non-null  uint16
 8   page_count         110984 non-null  uint16
 9   headers            110984 non-null  str   
 10  body               110984 non-null  str   
 11  athar_num          110984 non-null  int32 
 12  page               110984 non-null  str   
dtypes: int32(1), str(6), uint16(3), uint32(1), uint8(2)
memory usage: 125.0 MB


In [89]:
def optimize_int_columns(df):
    optimized_df = df.copy()
    for col in optimized_df.select_dtypes(include=['int64']).columns:
        min_val = optimized_df[col].min()
        max_val = optimized_df[col].max()

        if min_val >= 0:
            if max_val < 255: # uint8
                optimized_df[col] = optimized_df[col].astype('uint8')
            elif max_val < 65535: # uint16
                optimized_df[col] = optimized_df[col].astype('uint16')
            elif max_val < 4294967295: # uint32
                optimized_df[col] = optimized_df[col].astype('uint32')
        else:
            if min_val > -128 and max_val < 127: # int8
                optimized_df[col] = optimized_df[col].astype('int8')
            elif min_val > -32768 and max_val < 32767: # int16
                optimized_df[col] = optimized_df[col].astype('int16')
            elif min_val > -2147483648 and max_val < 2147483647: # int32
                optimized_df[col] = optimized_df[col].astype('int32')
    return optimized_df

# Apply integer optimization
full = optimize_int_columns(full)

In [92]:
full['book_name'] = full['book_name'].astype('category')
full['category_name'] = full['category_name'].astype('category')
full['all_authors'] = full['all_authors'].astype('category')
full['headers'] = full['headers'].astype('category')
full['page'] = full['page'].astype('category')

In [93]:
full.info()

<class 'pandas.DataFrame'>
Index: 110984 entries, 0 to 57928
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype   
---  ------             --------------   -----   
 0   book_id            110984 non-null  uint32  
 1   book_category      110984 non-null  uint8   
 2   book_type          110984 non-null  uint8   
 3   book_date          110984 non-null  uint16  
 4   book_name          110984 non-null  category
 5   category_name      110984 non-null  category
 6   all_authors        110984 non-null  category
 7   main_author_death  110984 non-null  uint16  
 8   page_count         110984 non-null  uint16  
 9   headers            110984 non-null  category
 10  body               110984 non-null  str     
 11  athar_num          110984 non-null  int32   
 12  page               110984 non-null  category
dtypes: category(5), int32(1), str(1), uint16(3), uint32(1), uint8(2)
memory usage: 98.9 MB


In [94]:
full.to_parquet('data/athar_books.parquet')

In [6]:
import os
from datasets import Dataset
dataset = Dataset.from_pandas(full)

# Push to your repository
dataset.push_to_hub("SHK4K/athar_books",token=os.environ["HF_TOKEN"])

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  3.08ba/s]
Processing Files (1 / 1): 100%|██████████| 39.2MB / 39.2MB,  594kB/s  
New Data Upload: 100%|██████████| 38.5MB / 38.5MB,  594kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [01:19<00:00, 79.46s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/SHK4K/athar_books/commit/45485066ce748518bb0aa43c13ae83b1f4ec9411', commit_message='Upload dataset', commit_description='', oid='45485066ce748518bb0aa43c13ae83b1f4ec9411', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/SHK4K/athar_books', endpoint='https://huggingface.co', repo_type='dataset', repo_id='SHK4K/athar_books'), pr_revision=None, pr_num=None)

In [ ]:
# sections_df_exp.to_parquet('data/sections_df_exp.parquet')
# sections_df.to_parquet('data/sections_df.parquet')

# see the df

In [3]:
df = pd.read_parquet('data/athar_books.parquet')
df.head(10)

,book_id,book_category,book_type,book_date,book_name,category_name,all_authors,main_author_death,page_count,headers,body,athar_num,page
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"١ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" مَنْ قَالَ: ا...",1,1
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"٢ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" إِذَا قَالَ ا...",2,2
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"٣ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" مَنْ قَالَ: ا...",3,2
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"٤ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" مَنْ قَالَ ذَ...",4,2
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,٥ - سَأَلْتُ أَبِي ﵀ عَنِ الصَّلَاةِ، خَلْفَ أ...,5,2
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,٦ - سَمِعْتُ أَبِي ﵀، يَقُولُ: «إِذَا كَانَ ال...,6,3
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,٧ - حَدَّثَنِي الْحَسَنُ بْنُ عِيسَى مَوْلَى ع...,7,4
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,٨ - حَدَّثَنِي مُحَمَّدُ بْنُ صَالِحٍ الْبَصْر...,8,5
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,٩ - حَدَّثَنِي أَحْمَدُ بْنُ إِبْرَاهِيمَ الدّ...,9,6
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,١٠ - حَدَّثَنِي أَحْمَدُ بْنُ سَعِيدٍ أَبُو جَ...,10,7


In [117]:
df[df['book_id']==17]

,book_id,book_category,book_type,book_date,book_name,category_name,all_authors,main_author_death,page_count,headers,body,athar_num,page


In [105]:
df['main_author_death'].unique()

array([ 290,  311,  256,  360,  241,  281,  643,  279,  387,  329, 1359,
        204,  399,  395,  198,  161,  255], dtype=uint16)

In [116]:
random = np.random.randint(0,len(df))
x = df.iloc[random]
display(Markdown(x['body']))
print(f'{x['book_id']}/{x['page']}')
print(x['book_name'])

٢٢٧ - حَدَّثَنَا الأَصَمُّ، قَالَ: أَخْبَرَنَا الرَّبِيعُ، قَالَ: أَخْبَرَنَا الْبُوَيْطِيُّ، قَالَ: أَخْبَرَنَا الشَّافِعِيُّ، قَالَ: أَخْبَرَنَا مُسْلِمٌ، وَعَبْدُ الْمَجِيدِ، قَالَ الرَّبِيعُ: أَحْسَبُهُ عَنِ ابْنِ جُرَيْجٍ، عَنْ مُوسَى بْنِ عُقْبَةَ، عَنْ عَبْدِ اللَّهِ بْنِ الْفَضْلِ، عَنِ الأَعْرَجِ، عَنْ عُبَيْدِ اللَّهِ بْنِ أَبِي رَافِعٍ، عَنْ عَلِيٍّ: أَنَّ النَّبِيَّ ﷺ كَانَ إِذَا رَكَعَ قَالَ: «اللَّهُمَّ لَكَ رَكَعْتُ وَبِكَ آمَنْتُ وَلَكَ أَسْلَمْتُ وَأَنْتَ رَبِّي، خَشَعَ لَكَ سَمْعِي وَبَصَرِي وَمُخِّي وَعِظَامِي وَمَا اسْتَقَلَّتْ بِهِ قَدَمِي للَّهِ رَبِّ الْعَالَمِينَ» . <<132>> 

9615/132
مسند الشافعي - ترتيب سنجر


# Embed

In [24]:
from datasets import load_dataset

ds = load_dataset('SHK4K/athar_books')
df = ds['train'].to_pandas()
df.head()

README.md:   0%|          | 0.00/936 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/39.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/110984 [00:00<?, ? examples/s]

,book_id,book_category,book_type,book_date,book_name,category_name,all_authors,main_author_death,page_count,headers,body,athar_num,page,__index_level_0__
0,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"١ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" مَنْ قَالَ: ا...",1,1,0
1,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"٢ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" إِذَا قَالَ ا...",2,2,0
2,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"٣ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" مَنْ قَالَ: ا...",3,2,0
3,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,"٤ - سَمِعْتُ أَبِي ﵀، يَقُولُ: "" مَنْ قَالَ ذَ...",4,2,0
4,323,1,1,290,السنة لعبد الله بن أحمد,العقيدة,عبد الله بن أحمد,290,1516,,٥ - سَأَلْتُ أَبِي ﵀ عَنِ الصَّلَاةِ، خَلْفَ أ...,5,2,0


In [25]:
from sentence_transformers import SentenceTransformer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentenceTransformer(
    "Omartificial-Intelligence-Space/GATE-AraBert-v1",
    device=device
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [26]:
for book in df['book_id'].unique():
    chunks_num = np.arange(1,len(df[df['book_id']==book])+1)
    df.loc[df['book_id']==book,'chunk_num'] = chunks_num

In [27]:
def book_meta(row):
    return {
        'book_name':row.book_name,
        'bood_date':row.book_date,
        'category_name': row.category_name,
        'authors': row.all_authors,
        'author_death': row.main_author_death,
        'athar_number': row.athar_num,
        'headers': row.headers,
        'page': row.page,
        'source': f"https://shamela.ws/book/{row.book_id}/{row.page.split(', ')[0]}"
    }

In [28]:
def clean(text):
    text = dediac_ar(text)
    text = normalize_alef_ar(text)
    text = text.replace('\u200f','')
    text = text.replace('  ', '')
    return text
    # return model([text])
df['body_clean'] = df['body'].progress_apply(clean)

100%|██████████| 110984/110984 [00:04<00:00, 23091.72it/s]


In [29]:
from tqdm import tqdm

book_ids,docs,metadatas = [],[],[]

for row in tqdm(df.itertuples(),total=len(df)):

    ids = str(row.book_category)+':'+str(row.book_id)+':'+str(int(row.chunk_num))
    assert ids not in book_ids, f"Duplicate id found: {ids}"
    docs.append(row.body_clean)

    metadata = book_meta(row)

    book_ids.append(ids)
    metadatas.append(metadata)




100%|██████████| 110984/110984 [01:02<00:00, 1769.26it/s]


In [30]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
embs = model.encode(
    docs,
    batch_size=768,      # tune this
    show_progress_bar=True,
    convert_to_numpy=True,
)

Batches:   0%|          | 0/217 [00:00<?, ?it/s]

In [36]:
torch.save(embs, 'sunnah.pt')

In [38]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [39]:
!cp sunnah.pt /content/drive/MyDrive/data

In [40]:
from google.colab import runtime
runtime.unassign()